# The portfolio view: every project, one MLflow server

This is the view for someone responsible for **many** projects: which model is in use for each one, how good it
is, and whether last month's checks passed. It reads **only** from the central MLflow server: it does not import
any project's code or open any project's files.

Before running it:
1. the server is running (README, Terminal 1);
2. the three projects have reported to it: `python malnutrition_pipeline.py`, `python tb_pipeline.py`,
   `python llm_pipeline.py` (README, Terminal 2).

Everything below uses three MLflow calls: `search_experiments`, `search_registered_models` and `search_runs`.
It works only because every project follows the same conventions (`hub.py`): the same tags, an `@champion` alias,
and monitoring runs with the same fields.

In [1]:
from pathlib import Path
import json

import mlflow
import pandas as pd
from IPython.display import HTML, display

import hub

client = hub.connect()                     # the central server: MLFLOW_TRACKING_URI, default http://127.0.0.1:5050
UI = hub.TRACKING_URI                      # the same address serves the web UI
pd.set_option("display.width", 200, "display.max_columns", 20, "display.max_colwidth", 80)
print("central server:", UI)

/home/user/mlops-workshop/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/10/29 06:03:45 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /home/user/mlops-workshop/.venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


central server: http://127.0.0.1:5050


## 1. What is on the server

One experiment per project. The tags say who owns it and its risk tier (the Practice Guide's tiers).

In [2]:
experiments = pd.DataFrame([
    {"experiment": e.name, **{t: e.tags.get(t, "") for t in hub.TAGS},
     "runs": len(client.search_runs([e.experiment_id], max_results=1000))}
    for e in client.search_experiments() if "project" in e.tags])
experiments.style.hide(axis="index")

experiment,project,use_case,owner,risk_tier,runs
llm-cds/guideline-assistant,llm-cds,guideline-assistant,Clinical decision support team,C,6
tb/cxr-screening,tb,cxr-screening,TB programme imaging team,C,4
malnutrition/gam-risk,malnutrition,gam-risk,Nutrition programme data team,C,8


## 2. The champion model of every project

For each registered model: the version with the `@champion` alias, the project's headline metric from the run that
produced it, and when it was promoted.

In [3]:
def champions():
    rows = []
    for rm in client.search_registered_models():
        if "champion" not in rm.aliases:
            continue
        mv = client.get_model_version_by_alias(rm.name, "champion")
        metric = rm.tags.get("headline_metric", "")
        rows.append({"project": rm.tags.get("project", ""), "registered model": rm.name,
                     "champion": f"v{mv.version}", "headline metric": metric,
                     "value": client.get_run(mv.run_id).data.metrics.get(metric),
                     "promoted": pd.to_datetime(mv.creation_timestamp, unit="ms").strftime("%Y-%m-%d %H:%M"),
                     "owner": rm.tags.get("owner", ""), "risk tier": rm.tags.get("risk_tier", ""),
                     "what it does": rm.description})
    return pd.DataFrame(rows).sort_values("project").reset_index(drop=True)


champions().style.format({"value": "{:.3f}"}).hide(axis="index")

project,registered model,champion,headline metric,value,promoted,owner,risk tier,what it does
llm-cds,llm-cds.guideline-assistant,v1,answer_rate,0.900,2026-10-29 06:03,Clinical decision support team,C,Answers health workers' questions from six national guidelines (RAG). Promotion rule: highest answer_rate (share of the 20 answerable questions not wrongly refused) among configurations with groundedness >= 0.9 that refuse all 4 out-of-scope questions.
malnutrition,malnutrition.gam-risk,v1,roc_auc,0.812,2026-10-29 06:02,Nutrition programme data team,C,"Risk of acute malnutrition (GAM) within 90 days, from routine EMR visits. The top 20% of visits by risk are followed up."
tb,tb.cxr-screening,v1,roc_auc,0.804,2026-10-29 06:03,TB programme imaging team,C,TB screening from chest X-ray image features; threshold set for 90% sensitivity at Kanyama. People flagged get a lab test.


The headline metrics are not comparable *between* projects (a ROC-AUC for X-rays, a correctness score for an
LLM). What the column shows is that every project has **named** the number it is judged on, and where to find it.

## 3. The latest checks of every project

Every monitoring run carries the same three fields, whichever tools the project used to decide them:

| Field | Values | Malnutrition decides it with | TB decides it with | LLM decides it with |
|---|---|---|---|---|
| `data_gate` | passed / blocked | Great Expectations (lesson 02's suite) | Great Expectations on the DICOM headers | fingerprint of the guideline documents |
| `drift` | ok / alert | Evidently (lessons 05–06) | a classifier telling sites apart | share of questions in everyday words |
| `performance` | ok / alert / pending | observed / predicted, 3 months late | sensitivity from lab results | pending: needs clinicians' review |

In [4]:
def monitoring_runs():
    runs = mlflow.search_runs(search_all_experiments=True, filter_string="tags.kind = 'monitoring'",
                              order_by=["attributes.start_time ASC"])
    out = pd.DataFrame({
        "project": runs["tags.project"], "period": runs["tags.period"],
        "data gate": runs["tags.data_gate"], "drift": runs["tags.drift"], "performance": runs["tags.performance"],
        "notes": runs.get("tags.notes", pd.Series(dtype=str)).fillna(""),
        "checked": runs["start_time"].dt.strftime("%Y-%m-%d %H:%M"),
        "run": [f'<a href="{UI}/#/experiments/{e}/runs/{r}" target="_blank">open</a>'
                for e, r in zip(runs["experiment_id"], runs["run_id"])],
        "run_id": runs["run_id"]})
    return out


def status(row):
    if row["data gate"] == "blocked" or row["performance"] == "alert":
        return "act now"
    if row["drift"] == "alert":
        return "look"
    return "ok"


COLOUR = {"act now": "#f6d5d0", "look": "#f8e3d0", "ok": "#e3f1ef"}


def paint(table):
    return (table.style.apply(lambda r: [f"background-color: {COLOUR[r['status']]}"] * len(r), axis=1)
            .hide(axis="index").set_properties(**{"text-align": "left"}))


checks = monitoring_runs()
checks["status"] = checks.apply(status, axis=1)
latest = checks.groupby("project").tail(1)                      # the most recent check of each project
paint(latest.drop(columns="run_id"))

project,period,data gate,drift,performance,notes,checked,run,status
malnutrition,2025-07,passed,ok,alert,"outcomes for 2025-04: observed/predicted 2.14, outside 0.67-1.5",2026-10-29 06:03,open,act now
tb,"site: Kanyama, HIV-positive",passed,ok,alert,"lab results: sensitivity 0.85, ROC-AUC 0.74",2026-10-29 06:03,open,act now
llm-cds,week 2,blocked,ok,pending,guideline documents changed since the champion was evaluated (ephi_cholera_2022.md): re-run the evaluation before using the new text,2026-10-29 06:03,open,act now


The rule for the colour is ours (`status` above): **act now** when data was blocked or performance is outside its
agreed range, **look** when inputs drifted, **ok** otherwise. Agree it once for the whole organisation.

## 4. One table for the person in charge

The champion and the latest check side by side: one row per project.

In [5]:
portfolio = (champions()[["project", "registered model", "champion", "headline metric", "value", "owner", "risk tier"]]
             .merge(latest[["project", "period", "data gate", "drift", "performance", "status", "run"]], on="project"))
paint(portfolio).format({"value": "{:.3f}"})

project,registered model,champion,headline metric,value,owner,risk tier,period,data gate,drift,performance,status,run
llm-cds,llm-cds.guideline-assistant,v1,answer_rate,0.900,Clinical decision support team,C,week 2,blocked,ok,pending,act now,open
malnutrition,malnutrition.gam-risk,v1,roc_auc,0.812,Nutrition programme data team,C,2025-07,passed,ok,alert,act now,open
tb,tb.cxr-screening,v1,roc_auc,0.804,TB programme imaging team,C,"site: Kanyama, HIV-positive",passed,ok,alert,act now,open


## 5. The history of every project

All monitoring runs, oldest first. This is where a pattern shows: the malnutrition model's performance alert in
July comes three months after food distribution stopped in April, while every check in between was quiet.

In [6]:
paint(checks.drop(columns=["run_id", "run"]))

project,period,data gate,drift,performance,notes,checked,status
malnutrition,2025-01,passed,ok,pending,,2026-10-29 06:02,ok
malnutrition,2025-02,blocked,ok,pending,"blocked by the data gate: expect_column_values_to_be_unique(visit_id), expect_column_values_to_be_in_set(sex), expect_column_values_to_be_between(age_months)",2026-10-29 06:02,act now
malnutrition,2025-03,passed,alert,pending,drift: Value drift for risk; Value drift for region,2026-10-29 06:02,look
malnutrition,2025-04,passed,ok,ok,,2026-10-29 06:02,ok
malnutrition,2025-05,passed,ok,pending,"no outcomes to check: 2025-02 was blocked, never scored",2026-10-29 06:02,ok
malnutrition,2025-06,passed,ok,ok,,2026-10-29 06:02,ok
malnutrition,2025-07,passed,ok,alert,"outcomes for 2025-04: observed/predicted 2.14, outside 0.67-1.5",2026-10-29 06:03,act now
tb,site: Chawama,passed,ok,alert,"lab results: sensitivity 0.61, ROC-AUC 0.74",2026-10-29 06:03,act now
tb,site: Chainda South,blocked,alert,alert,"image headers differ from Kanyama: expect_column_values_to_be_in_set(Manufacturer), expect_column_values_to_be_in_set(PhotometricInterpretation), expect_column_values_to_be_in_set(PixelSpacing), expect_column_values_to_be_in_set(Columns); lab results: sensitivity 0.00, ROC-AUC 0.65",2026-10-29 06:03,act now
tb,"site: Kanyama, HIV-positive",passed,ok,alert,"lab results: sensitivity 0.85, ROC-AUC 0.74",2026-10-29 06:03,act now


## 6. Drill down: the evidence is on the server too

Each monitoring run keeps the tools' own reports as artifacts (`reports/`): the full Great Expectations result
(JSON) and, where used, the Evidently report (HTML). Anyone with access to the server can read them, without
access to the project's machine. Here: why was the malnutrition data blocked in February?

In [7]:
blocked = checks[(checks["project"] == "malnutrition") & (checks["data gate"] == "blocked")].iloc[0]
files = [a.path for a in client.list_artifacts(blocked["run_id"], "reports")]
print("reports kept with that run:", files)

gx_result = json.loads(Path(mlflow.artifacts.download_artifacts(
    run_id=blocked["run_id"], artifact_path=next(f for f in files if f.endswith(".json")))).read_text())
failed = [{"expectation": r["expectation_config"]["type"],
           "column": r["expectation_config"]["kwargs"].get("column", "table"),
           "severity": r["expectation_config"].get("severity", ""),
           "rows failing": r["result"].get("unexpected_count", ""),
           "observed": r["result"].get("observed_value", "")}
          for r in gx_result["results"] if not r["success"]]
print(f"{gx_result['statistics']['unsuccessful_expectations']} of {gx_result['statistics']['evaluated_expectations']} "
      f"expectations failed in {blocked['period']}:")
pd.DataFrame(failed)

reports kept with that run: ['reports/gx_2025-02.json']
6 of 42 expectations failed in 2025-02:


,expectation,column,severity,rows failing,observed
0,expect_column_values_to_be_unique,visit_id,critical,30,
1,expect_column_values_to_be_in_set,sex,critical,41,
2,expect_column_values_to_be_between,age_months,critical,12,
3,expect_column_values_to_be_between,muac_cm,critical,420,
4,expect_column_values_to_be_null,muac_cm,warning,79,
5,expect_column_min_to_be_between,whz,critical,,-2.99


In [8]:
drifted = checks[(checks["project"] == "malnutrition") & (checks["drift"] == "alert")].iloc[0]
report = next(a.path for a in client.list_artifacts(drifted["run_id"], "reports") if a.path.endswith(".html"))
local = mlflow.artifacts.download_artifacts(run_id=drifted["run_id"], artifact_path=report, dst_path="data/downloads")
print(f"{drifted['period']}: {drifted['notes']}")
print("The Evidently report, downloaded from the server; open it in a browser:", Path(local).resolve())
display(HTML(f'Or in the MLflow UI: <a href="{UI}/#/experiments/{mlflow.get_run(drifted["run_id"]).info.experiment_id}'
             f'/runs/{drifted["run_id"]}/artifacts/{report}" target="_blank">the run\'s Artifacts tab</a> '
             f'(the report is large: use the download button there).'))

2025-03: drift: Value drift for risk; Value drift for region
The Evidently report, downloaded from the server; open it in a browser: /home/user/mlops-workshop/central-mlflow/data/downloads/evidently_2025-03.html


## 7. Questions across all projects

Because the tags are shared, one search answers questions about the whole organisation. The same filter strings
work in the search box of the MLflow UI.

In [9]:
questions = {
    "Every blocked data gate, in any project": "tags.kind = 'monitoring' and tags.data_gate = 'blocked'",
    "Tier C projects with a performance alert": "tags.risk_tier = 'C' and tags.performance = 'alert'",
    "Everything the TB team logged": "tags.owner = 'TB programme imaging team'",
}
for question, query in questions.items():
    found = mlflow.search_runs(search_all_experiments=True, filter_string=query)
    print(f"{question}\n    {query}\n    -> {len(found)} runs: "
          + ", ".join(f"{p} {n}" for p, n in zip(found["tags.project"], found["tags.mlflow.runName"])) + "\n")

Every blocked data gate, in any project
    tags.kind = 'monitoring' and tags.data_gate = 'blocked'
    -> 3 runs: llm-cds monitoring week 2, tb monitoring site: Chainda South, malnutrition monitoring 2025-02

Tier C projects with a performance alert
    tags.risk_tier = 'C' and tags.performance = 'alert'
    -> 4 runs: tb monitoring site: Kanyama, HIV-positive, tb monitoring site: Chainda South, tb monitoring site: Chawama, malnutrition monitoring 2025-07

Everything the TB team logged
    tags.owner = 'TB programme imaging team'
    -> 4 runs: tb monitoring site: Kanyama, HIV-positive, tb monitoring site: Chainda South, tb monitoring site: Chawama, tb train Kanyama, HIV-negative



## 8. The same view in the MLflow UI

Open the server's address (http://127.0.0.1:5050) and switch the toggle at the top left to **Model training**.

- **Experiments**: one per project (`malnutrition/gam-risk`, `tb/cxr-screening`, `llm-cds/guideline-assistant`).
  Tick all three and you see every run of every project in one table; add the columns `project`, `period`,
  `data_gate`, `drift`, `performance` from the tags.
- **Models**: every registered model with its `@champion` alias, owner and risk tier.
- **A monitoring run → Artifacts → `reports/`**: the Great Expectations result (shown in the browser) and the
  Evidently report (large: download it with the button at the top right of the preview).

## What this does not do (yet)

- **Who may see and change what.** Here everyone can do everything. A real server needs logins and
  permissions (README, "From the local remote to a real one").
- **Notifications.** This is a view, not an alarm. Alerts come from each project's pipeline (Prefect lesson),
  which can also post to a chat group or email.
- **Decisions.** The server records who promoted what and when. The decision itself, with its evidence and the
  name of the person who signed it, belongs in the Gate 2 record of the Practice Guide.